In [1]:
import PyPDF2
import os
from pathlib import Path
import logging

# OCR specific imports
try:
    from PIL import Image
except ImportError:
    Image = None
try:
    import pytesseract
except ImportError:
    pytesseract = None
try:
    from pdf2image import convert_from_path
except ImportError:
    convert_from_path = None

from permit_data_extraction.config import RAW_DATA_DIR, INTERIM_DATA_DIR

2025-05-15 15:57:29.262 | INFO     | permit_data_extraction.config:<module>:11 - PROJ_ROOT path is: /home/afhubbard/permit_data_extraction


In [2]:
# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
# Directory containing your PDF permit files
PDF_INPUT_DIR = Path(RAW_DATA_DIR)

# Directory where extracted .txt files will be saved
TEXT_OUTPUT_DIR = Path(INTERIM_DATA_DIR)

In [8]:
TESSERACT_CMD = None # Set this if needed

In [9]:
def ocr_pdf_pages(pdf_path, poppler_path=None):
    """
    Performs OCR on each page of a PDF file.
    Requires pdf2image and pytesseract.
    `poppler_path` is optional if poppler is not in system PATH.
    """
    if not pytesseract or not convert_from_path or not Image:
        logging.error("OCR libraries (pytesseract, pdf2image, Pillow) not available. Skipping OCR for this file.")
        return None

    ocr_text_content = ""
    try:
        logging.info(f"    Attempting OCR for {pdf_path.name}...")
        images = convert_from_path(pdf_path, poppler_path=poppler_path)
        for i, image in enumerate(images):
            logging.info(f"      OCRing page {i+1} of {len(images)} for {pdf_path.name}...")
            try:
                page_text_content = pytesseract.image_to_string(image, lang='eng') # Specify language
                ocr_text_content += page_text_content + "\n\n--- Page Break ---\n\n" # Add page break marker
            except pytesseract.TesseractNotFoundError:
                logging.error(
                    "Tesseract is not installed or not in your PATH. "
                    "Please install Tesseract and make sure it's accessible. "
                    "If installed, you might need to set TESSERACT_CMD at the top of the script."
                )
                return None # Stop OCR if Tesseract is not found
            except Exception as ocr_page_e:
                logging.warning(f"      Error OCRing page {i+1} of {pdf_path.name}: {ocr_page_e}")
        logging.info(f"    OCR completed for {pdf_path.name}. Extracted approx {len(ocr_text_content)} chars.")
        return ocr_text_content
    except Exception as e:
        logging.error(f"    Error during OCR process for {pdf_path.name}: {e}", exc_info=True)
        if "Unable to get page count" in str(e) or "pdfinfo" in str(e):
            logging.error("    This might be due to Poppler not being installed or not found in PATH.")
            logging.error("    Please ensure Poppler utilities are installed and accessible.")
        return None

In [10]:
def extract_text_from_single_pdf(pdf_path):
    """
    Extracts text from a single PDF file.
    First tries direct text extraction with PyPDF2.
    If direct extraction yields minimal text, falls back to OCR.
    """
    direct_text_content = ""
    num_pages = 0
    extraction_method_used = "PyPDF2 (Direct)"

    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            num_pages = len(reader.pages)
            if num_pages == 0:
                logging.warning(f"  {pdf_path.name} has 0 pages according to PyPDF2.")
                # Try OCR anyway if PyPDF2 says 0 pages, as it might be an image PDF
                if pytesseract and convert_from_path and Image:
                    logging.info(f"  Attempting OCR for {pdf_path.name} as PyPDF2 reported 0 pages.")
                    poppler_bin_path = None # Set this if pdf2image has issues finding poppler
                    ocr_fallback_text = ocr_pdf_pages(pdf_path, poppler_path=poppler_bin_path)
                    if ocr_fallback_text:
                        extraction_method_used = "OCR (PyPDF2 reported 0 pages)"
                        return ocr_fallback_text, extraction_method_used
                return None, "PyPDF2 (0 pages, OCR not attempted or failed)"


            logging.info(f"  Reading {num_pages} pages from {pdf_path.name} using PyPDF2...")
            for page_num in range(num_pages):
                try:
                    page = reader.pages[page_num]
                    page_text_content = page.extract_text()
                    if page_text_content:
                        direct_text_content += page_text_content + "\n\n--- Page Break ---\n\n" # Add page break marker
                except Exception as page_e:
                    logging.warning(f"    Could not extract text from page {page_num + 1} with PyPDF2 in {pdf_path.name}: {page_e}")
        logging.info(f"  PyPDF2 extracted approx {len(direct_text_content)} chars from {pdf_path.name}.")

        min_chars_threshold = num_pages * 50 # Heuristic: e.g., < 50 chars per page on average
        if num_pages > 0 and (not direct_text_content or len(direct_text_content.replace("--- Page Break ---", "").strip()) < min_chars_threshold):
            logging.info(f"  Direct text extraction from {pdf_path.name} yielded minimal text. Attempting OCR as fallback.")
            poppler_bin_path = None # Set this if pdf2image has issues finding poppler
            ocr_text_content = ocr_pdf_pages(pdf_path, poppler_path=poppler_bin_path)
            if ocr_text_content:
                logging.info(f"  Using OCR text for {pdf_path.name}.")
                extraction_method_used = "OCR (Fallback)"
                return ocr_text_content, extraction_method_used
            else:
                logging.warning(f"  OCR failed or produced no text for {pdf_path.name}. Using PyPDF2 text (if any).")
                return direct_text_content if direct_text_content else None, extraction_method_used if direct_text_content else "PyPDF2 (OCR Failed)"
        elif not direct_text_content and num_pages == 0: # Should be caught above, but as a safeguard
            logging.warning(f"  {pdf_path.name} appears to have 0 pages or is unreadable by PyPDF2.")
            return None, "PyPDF2 (0 pages)"
        else:
            logging.info(f"  Using direct text extraction (PyPDF2) for {pdf_path.name}.")
            return direct_text_content, extraction_method_used

    except FileNotFoundError:
        logging.error(f"  File not found at {pdf_path}")
        return None, "File Not Found"
    except PyPDF2.errors.PdfReadError as pdf_err:
        logging.error(f"  Error reading PDF {pdf_path.name} with PyPDF2 (possibly corrupted or password-protected): {pdf_err}")
        logging.info(f"  Attempting OCR as PyPDF2 failed for {pdf_path.name}.")
        poppler_bin_path = None # Set this if pdf2image has issues finding poppler
        ocr_text_content = ocr_pdf_pages(pdf_path, poppler_path=poppler_bin_path)
        extraction_method_used = "OCR (PyPDF2 ReadError)" if ocr_text_content else "PyPDF2 ReadError (OCR Failed)"
        return ocr_text_content, extraction_method_used
    except Exception as e:
        logging.error(f"  Unexpected error reading PDF {pdf_path.name}: {e}", exc_info=True)
        return None, "Unexpected Error"

In [11]:
def save_text_to_file(text_content, output_path):
    """Saves the given text content to a file."""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(text_content)
        logging.info(f"    Successfully saved extracted text to {output_path}")
    except Exception as e:
        logging.error(f"    Error saving text to {output_path}: {e}", exc_info=True)

In [12]:
logging.info("Starting PDF to Text Conversion Process...")

# Check and set Tesseract command if specified
if TESSERACT_CMD and pytesseract:
    logging.info(f"Setting pytesseract.tesseract_cmd to: {TESSERACT_CMD}")
    pytesseract.tesseract_cmd = TESSERACT_CMD
elif not pytesseract and TESSERACT_CMD:
    logging.warning("TESSERACT_CMD is set, but pytesseract library is not imported/available.")


if not PDF_INPUT_DIR.is_dir():
    logging.critical(f"Error: PDF input directory not found at '{PDF_INPUT_DIR}'. Please create it and add PDF files.")


if not TEXT_OUTPUT_DIR.exists():
    logging.info(f"Creating text output directory at '{TEXT_OUTPUT_DIR}'")
    TEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = list(PDF_INPUT_DIR.glob('*.pdf'))
if not pdf_files:
    logging.warning(f"No PDF files found in '{PDF_INPUT_DIR}'. Ensure files end with '.pdf'.")


logging.info(f"Found {len(pdf_files)} PDF files to process.")
summary = []

for pdf_path in pdf_files:
    logging.info(f"\nProcessing file: {pdf_path.name}")
    text_content, method = extract_text_from_single_pdf(pdf_path)

    if text_content:
        output_filename = pdf_path.stem + ".txt" # e.g., mypermit.pdf -> mypermit.txt
        output_file_path = TEXT_OUTPUT_DIR / output_filename
        save_text_to_file(text_content, output_file_path)
        summary.append({"filename": pdf_path.name, "status": "Success", "method": method, "output_file": str(output_file_path)})
    else:
        logging.warning(f"  No text extracted from {pdf_path.name}. Method: {method}")
        summary.append({"filename": pdf_path.name, "status": "Failed", "method": method, "output_file": None})

logging.info("\n--- Processing Summary ---")
for item in summary:
    logging.info(f"File: {item['filename']}, Status: {item['status']}, Method: {item['method']}, Output: {item['output_file']}")
logging.info("PDF to Text conversion process finished.")

: 

In [5]:
import ocrmypdf

In [4]:
pdf_files = list(PDF_INPUT_DIR.glob('*.pdf'))

In [6]:
pdf_files

[PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Procter & Gamble [p38].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Fontana Paper Mills [p10].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Air Products and Chemicals Inc [p11].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Phillips 66 Refinery [p22].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Valero Benicia Asphalt Plant [p8].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Dow Chemical [p10].pdf'),
 PosixPath('/home/afhubbard/permit_data_extraction/data/raw/Anheuser Busch.pdf')]

In [7]:
for file in pdf_files:
    output_file = TEXT_OUTPUT_DIR / file.name
    if not output_file.exists():
        logging.info(f"Converting {file} to {output_file}")
        ocrmypdf.ocr(file, output_file, deskew=True, rotate_pages=True)
    else:
        logging.info(f"File {output_file} already exists. Skipping OCR.")

ERROR:ocrmypdf.subprocess:
The program 'gs' could not be executed or was not found on your
system PATH.



MissingDependencyError: The program 'gs' did not report its version. Message was:
gs, version 1.1.1
